# Preprocessing & Tokenization

## Load the dataset into a hugging face dataset object

In [ ]:
import datasets


train_set = datasets.load_dataset("csv", data_files="../data/train_set.csv")['train']
train_set

In [ ]:
train_set[:3]['comment_text']

## Preprocessing

In [ ]:
import re

# existing patterns
username_pattern = re.compile(r'\[\[user.*?\]\]|\[\[user talk.*?\]\]|user:\w+', re.IGNORECASE)
whitespace_pattern = re.compile(r'\s+')

# wiki markup patterns
wiki_link_pattern = re.compile(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]')
wiki_template_pattern = re.compile(r'\{\{.*?\}\}')
wiki_bold_italic_pattern = re.compile(r"'{2,5}")
wiki_heading_pattern = re.compile(r'=+\s*(.*?)\s*=+')

def preprocess_batch(batch):
    cleaned_texts = []

    for text in batch["comment_text"]:
        text = text.lower()

        # remove usernames
        text = username_pattern.sub(" ", text)

        # convert [[link|text]] -> text
        text = wiki_link_pattern.sub(r"\1", text)

        # remove templates {{...}}
        text = wiki_template_pattern.sub(" ", text)

        # remove bold/italic markup
        text = wiki_bold_italic_pattern.sub("", text)

        # remove headings
        text = wiki_heading_pattern.sub(r"\1", text)

        # normalize whitespace
        text = whitespace_pattern.sub(" ", text).strip()

        cleaned_texts.append(text)

    batch["comment_text"] = cleaned_texts
    return batch

In [ ]:
train_set = train_set.map(preprocess_batch, batched=True, num_proc=4)

In [ ]:
train_set[:3]['comment_text']